# Phase 3: A/B Testing 面试收割机 🧪

> **🎯 目标**: 把你的实战经验（某跨境电商S公司/Wind）转化为大厂（某短视频大厂B公司/某新能源车企T公司）能听懂的“黑话”。
> **Status**: `User: Experienced`, `Goal: Theory Systematization`

## 1. 甚至是资深分析师也容易搞混的概念 🤯

面试官问：“什么是 Alpha 和 Beta？什么是 Power？”
不要背书，请用“人话”解释：

*   **Alpha (Type I Error)**: **“冤枉好人”**的概率。
    *   *场景*: 实验其实没效果，但你大喊“有显著提升！”
    *   *标准*: 通常设为 0.05 (5%)。
*   **Beta (Type II Error)**: **“放过坏人”**的概率。
    *   *场景*: 实验其实有效果，但你没检测出来，错过了涨薪的机会。
    *   *标准*: 通常设为 0.20 (20%)。
*   **Power (1 - Beta)**: **“坏人一抓一个准”**的概率。
    *   *标准*: 1 - 0.20 = 0.80 (80%)。我们希望原本有效的实验，我们有 80% 的把握能验出来。
*   **MDE (Minimum Detectable Effect)**: **“值得抓的坏人有多坏”**。
    *   *场景*: 如果提升只有 0.0000001%，老板不关心。我们只关心比如 1% 以上的提升。
    *   *规律*: MDE 越小（想检测微小的提升），需要的样本量 (Sample Size) 就越大（大海捞针）。

## 2. 实战：用 Python 计算样本量 (Sample Size) 🧮

别再用网上的在线计算器了，面试时要手写代码算的。
我们会用到 `statsmodels` 这个库（比 scipy 更专用于统计）。

In [ ]:
import numpy as np
import statsmodels.stats.api as sms

# 场景: 我们现在的转化率 (Baseline Conversion Rate) 是 10% (0.10)
# 目标: 我们想检测出至少 2% 的绝对提升 (比如从 10% 提升到 12%)

# 1. Effect Size (效应量): 这里用的是 Cohen's h
# p1 = Baseline (当前水平): 比如现在页面转化率是 10%
p1 = 0.10
# p2 = Target (目标水平): 我们希望新版本能达到 12% (即提升 2pp)
p2 = 0.12

# ⚠️ 面试坑点: Cohen's h != (p2 - p1) 也不是 p2/p1
# Cohen's h = 2 * (arcsin(sqrt(p1)) - arcsin(sqrt(p2)))
# 为什么要搞这么复杂？因为 "方差" 不一样。
# - 在 50% 附近的提升最难 (方差最大)
# - 在 1% 或 99% 附近的提升较易 (方差小)
# 这个函数把不同起点的难度 "拉平" 了。
# ⚠️ 注意: effect_size 不是 0-1 之间的数！它可以大于 1。
# 经验法则 (Jacob Cohen):
# - 0.2: 小效应 (Small) -> 需要巨大样本量
# - 0.5: 中效应 (Medium)
# - 0.8: 大效应 (Large) -> 这种提升一眼就能看出来，不需要太多样本
#
# 💡 关键区别:
# - 给老板汇报用 Relative Risk (RR): "老板，我们提升了 20%!" (p2/p1 - 1)
# - 给机器算样本量用 Cohen's h: 因为机器需要消除方差差异。
effect_size = sms.proportion_effectsize(p1, p2)

In [ ]:
# 🧪 Math Corner (选修): 为什么要用 Arcsin?
# 让我们画图看看: 一般的比例 p，它的方差是 p*(1-p)。
# 这是一个倒 U 型曲线: 在 0.5 时方差最大(最难测)，在 0 和 1 附近方差小。
# Cohen's h 通 Arcsin 变换，把这个倒 U 型 "拉直" 了，让方差在所有地方都一样。

import matplotlib.pyplot as plt

x = np.linspace(0.01, 0.99, 100)
variance = x * (1 - x)
transformed = 2 * np.arcsin(np.sqrt(x))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(x, variance, label='Raw Variance: p(1-p)')
plt.title("Raw Proportion Variance (倒U型)")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(x, np.gradient(transformed), label='Transformed Gradient')
plt.title("Transformed Gradient (恒定/平稳)")
plt.ylim(0, 5)
plt.legend()
plt.show()

In [ ]:
# 2. 算样本量
required_n = sms.NormalIndPower().solve_power(
    effect_size=effect_size, 
    power=0.8, 
    alpha=0.05, 
    ratio=1  # 实验组和对照组 1:1 分配
)

print(f"我们需要每组至少 {int(np.ceil(required_n))} 个样本。")
# 思考: 如果把 p2 改成 0.101 (只提升 0.1%)，样本量会爆炸吗？试一下！

## 3. 面试硬核题：Ratio Metric (如 CTR) 的坑 🕳️

你提到的 **Delta Method** 是对的！

**问题**: 普通的 T-test 假设每个样本是独立的。但是 CTR (Click-Through-Rate) = `Sum(Clicks) / Sum(Views)`。
分子分母都随机变化，而且同一个用户可能贡献多次 View，数据不独立 (i.i.d 假设失效)。

**面试回答模板**：
> "对于 CTR 这种 Ratio 指标，直接算方差会低估。工业界通常有两种解法：
> 1. **Delta Method**: 用泰勒展开近似计算方差（适合大数据量，工程好实现）。
> 2. **Bootstrap**: 暴力重采样算方差（准确但算得慢）。
> 3. **用户层级汇总**: 简单的做法是先把数据 aggregated 到 user level，算出每个用户的点击率，然后对用户均值做 T-test。（最常用）"

## 🏋️‍♂️ 模拟面试挑战 (Challenge)

**背景**: 某短视频大厂B公司某新功能上线。
**数据**:
*   **Control (对照组)**: 1000 人，转化 100 人。
*   **Treatment (实验组)**: 1000 人，转化 125 人。

**任务**:
1. 计算两组的转化率。
2. 这是一个显著的提升吗？(P-value < 0.05?)
3. 用 `statsmodels` 的 `proportions_ztest` 告诉我答案。

In [11]:
from statsmodels.stats.proportion import proportions_ztest

# 成功次数 (Conversions)
count = np.array([125, 100]) # [实验组, 对照组]

#通过样本数 (Observations)
nobs = np.array([1000, 1000])

# ✍️ 在这里写代码计算 P-value
# stat, p_val = proportions_ztest(..., ...)
# print(f"P-value: {p_val:.4f}")

# -----------------------------------------------------------
# 🧠 4. 终极指南: 什么时候用什么检验？(Cheat Sheet)
# -----------------------------------------------------------
# 面试时，先看由于 "数据类型" 决定:
#
# 1. **均值类 (Means)**: 比如 "人均时长", "客单价"。
#    - 数据: 连续数值 (Continuous)。
#    - 武器: **T-test** (`scipy.stats.ttest_ind`)。
#    - *冷知识: 样本量 > 30 时，Z-test 和 T-test 结果其实差不多。但在工业界主要用 T-test。*
#
# 2. **比例类 (Proportions)**: 比如 "转化率" (用户级)。
#    - 定义: 分母 = Randomization Unit (通常是 Unique Users)。
#    - 特点: 每个用户只有两个结局 (买/没买)。就是一个硬币扔一次。
#    - 武器: **Z-test** (`proportions_ztest`)。
#
# 3. **比率类 (Ratios)**: 比如 "CTR" (点击/曝光)。
#    - 定义: 分母 = Events (比如 Views)。
#    - 特点: 一个用户可能有 100 次 View。这 100 次 View 是**不独立**的 (同一个人的习惯)。
#    - 武器: **Delta Method**。
#
# 💡 **判别金标准**:
# 问自己: 分母是谁？
# - 分母是 "人" (去重用户) -> 用 Z-test。
# - 分母是 "事" (次数) -> 用 Delta Method。
#
# -----------------------------------------------------------
# 🌟 Star Case: 智能客服转人工率 (您的真实案例)
# -----------------------------------------------------------
# 场景: 必须按 "用户" 分流 (保持体验一致)，但指标是 "会话级转人工率" (Session Resolution Rate)。
# - 分子: 转人工 Session 数
# - 分母: 总 Session 数
#
# 判定:
# 1. Randomization Unit = User
# 2. Analysis Unit = Session
# 3. 出现 "Unit Mismatch" (单位不一致) -> 一个用户产生多个 Session (不独立)。
#
# ✅ 结论: 你们当年用 Delta Method 是完全正确的！
# 如果用了 Z-test，就会低估方差，犯 Type I Error。
# 这就是面试时的 "高光时刻" (Highlight).

# -----------------------------------------------------------

# 🔓 附加题: 既然 P=0.076 (不显著)，那到底要多少样本才显著？
# 我们用刚才学的 solve_power 算一下:
# p1 = 0.10, p2 = 0.125
effect_size_challenge = sms.proportion_effectsize(0.10, 0.125)

required_n_challenge = sms.NormalIndPower().solve_power(
    effect_size=effect_size_challenge, 
    power=0.8, 
    alpha=0.05, 
    ratio=1
)

print(f"原来如此！如果你想检测出这 25% 的提升（Target=12.5%），每组至少需要 {int(np.ceil(required_n_challenge))} 人。")
print(f"现在的 1000 人还差得远呢！这解释了为什么 P > 0.05 (Power 不足)。")

原来如此！如果你想检测出这 25% 的提升（Target=12.5%），每组至少需要 2501 人。
现在的 1000 人还差得远呢！这解释了为什么 P > 0.05 (Power 不足)。
